# Iterators and Generators

https://docs.python.org/3/tutorial/classes.html#iterators

## Iterators

The two definitions we need to wrap our heads around today are...

*Iterable:*
 - an object (container) that can be iterated over by a `for` loop, e.g. lists, tuples, sets, and dictionaries;
 - has an `__iter__` method that returns an iterator.

*Iterator:*
 - the companion object of an iterable
 - (often) its purpose is to enable a `for` loop over an iterable;
 - has a `__next__` method returning the next element of the iteration.

In [1]:
# lists are iterable
L = [1,2,3]
for e in L:
    print(e)

1
2
3


Behind the scenes:
1. call iter(L) to get an iterator object, let's call it tmp_iterator_object
2. call next(tmp_iterator_object) to access element one at a time
3. stop once the Exception StopIteration is raised

In [2]:
L = [1,2,3]
tmp_iter_object = iter(L)
print(type(tmp_iter_object), type(L))

<class 'list_iterator'> <class 'list'>


In [3]:
while True:
    try:
        print(next(tmp_iter_object))
    except StopIteration:
        break

1
2
3


Iterators are something you can call `__next__` on.

For L to be iterable ('for-loop-able'), 
1. L needs to have __iter__ implemented so that iter(L) returns an iterator
2. iterator returned by iter(L) needs to have __next__ implemented



However, not every container-like object automatically supports iteration. For example, here's a simple class that holds a list and...doesn't do much else.


In [8]:
class boringList:
    def __init__(self, L):
        self.L = L
    
    def __str__(self):
        return str(self.L)

In [9]:
B = boringList([1,2,3])
print(B, type(B))


[1, 2, 3] <class '__main__.boringList'>


In [10]:
class boringList:
    def __init__(self, L):
        self.L = L
    
    def __str__(self):
        return str(self.L)
    
    def __iter__(self):
        return iter(self.L)

In [11]:
B = boringList([1,2,3])
for e in B:
    print(e)

1
2
3


So, what is needed to iterate over a container? Briefly, we need to define an iterator class that implements a __next__() method. 

To get the iterator from the container, we need to define an __iter__() method that tells Python how to construct an appropriate iterator, with the required __next__() method. 

In [15]:
# Let's try making our boringList iterable. 
# We need an __iter__() method and a __next__() method. 
# To make the workings of the code transparent, we'll add a few print statements.
class boringList:
    def __init__(self, L):
        self.L = L

    def __str__(self):
        return str(self.L)

    def __iter__(self):
        print('boringList.__iter__() called')
        return boringListIterator(self)

class boringListIterator:
    def __init__(self, bL):
        print('boringListIterator object created')
        self.L = bL.L
        self.i = 0      # keep track of curr index

    def __next__(self):
        print('boringListIterator.__next__() called')
        if self.i < len(self.L):
            element = self.L[self.i]
            self.i +=1
            return element
        else:
            print ('StopIteration')
            raise StopIteration

In [16]:
B = boringList([1,2,3])
for e in B:
    print(e)

boringList.__iter__() called
boringListIterator object created
boringListIterator.__next__() called
1
boringListIterator.__next__() called
2
boringListIterator.__next__() called
3
boringListIterator.__next__() called
StopIteration


Behind the scenes, the for keyword calls iter() on B, creating an object of type boringListIterator. 

Because this object has a __next__() method, the for loop can then operate as expected. 

In [17]:
# Here's the "slow-motion" version:

# create an iterator
BLI = iter(B)

# repeated until the StopIteration exception is raised. 
while True:
    try: 
        print(next(BLI))
    except StopIteration:
        break

boringList.__iter__() called
boringListIterator object created
boringListIterator.__next__() called
1
boringListIterator.__next__() called
2
boringListIterator.__next__() called
3
boringListIterator.__next__() called
StopIteration


Another approach to the same problem is to define a __next__() method for boringList itself. 

In this case, the __iter__() method can simply return self. The __iter__() method should reset i to 0 each time to ensure that we always loop starting from the beginning.

In [20]:
class boringList:
    def __init__(self,L):
        self.L = L

    def __str__(self):
        return str(self.L)
    
    def __iter__(self):
        self.i = 0
        return self

    def __next__(self):
        if self.i < len(self.L):
            self.i +=1 
            return self.L[self.i-1]
        
        else:
            raise StopIteration
    

In [21]:
# now this again works
B = boringList([1,2,3])
for e in B:
    print(e)

1
2
3


**Question**: What will happen if I move `self.i = 0` to a different place, say inside `__init__`?

In [22]:
B = boringList([1,2,3])
for e in B:
    print(e)

1
2
3


In [23]:
for e in B:
    print(e)

1
2
3


## Generators

A **generator** is a powerful syntactic shortcut for creating iterators. 

Rather than writing a whole customized iterator class, a generator allows you to construct iterators using function-like syntax. The key syntactic difference between functions and generators is the `yield` keyword, which replaces `return` in functions. There are also several important semantic differences. Let's look at an example.  

In [24]:
def every_other_generator(data):
    for i in range(0, len(data), 2):
        yield data[i]

#equivalent to 
# for j in range(0, len(data)//2):
#     yield data[2*j]

In [25]:
data = '0123456789'
Gen = every_other_generator(data)

In [26]:
print(type(Gen))

<class 'generator'>


In [27]:
print(Gen)

<generator object every_other_generator at 0x111cf8400>


In [28]:
for e in Gen:
    print(e)

0
2
4
6
8


In [29]:
class BoringList:
    def __init__(self,L):
        self.L = L
        
    def __str__(self):
        return str(self.L)
    
    def bl_generator(self):
        for e in self.L:
            yield e

In [31]:
B = BoringList([1,2,3])
for e in B.bl_generator():
    print(e)

1
2
3


In [33]:
# Another example
import random
def while_loop_gen(data):
    iter_num  = 0
    while iter_num < 5:
        i = random.choice(range(len(data)))
        iter_num +=1 
        yield data[i]

In [35]:
data = '0123456789'
Gen = while_loop_gen(data)
print(type(Gen))

for e in Gen:
    print(e)

<class 'generator'>
3
8
3
7
2
